In [12]:
import imageio
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import pymongo
from datetime import datetime
import moviepy.editor as mp
import diff_match_patch as dmp_module
import re
dmp = dmp_module.diff_match_patch()

from tqdm import tqdm

In [57]:
coloradd = '#82E0AA' 
colorDel = '#F1948A'

image_size = (866, 1080)

def imageHighlight(total):
    add = coloradd
    delete = colorDel

    font = ImageFont.truetype(".\ARIAL.ttf", 16)
    image = Image.new("RGB", image_size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    y_text = 18
    x_text = 20
    width = 111
    char_num = 0
    for segment in total:
        for char in segment[1]:
            if char_num >= width or char=='\n':
                char_num = 0
                y_text += 18
                x_text = 20
            if segment[0] == 0:
                draw.text((x_text, y_text), char, font=font, fill=(0, 0, 0))
            elif segment[0] == -1:
                draw.rectangle(((x_text, y_text),
                                (x_text + draw.textsize(char, font)[0],y_text + 15)), fill=delete)
                draw.text((x_text, y_text), char, font=font, fill=(0, 0, 0))
            elif segment[0] == 1:
                draw.rectangle(((x_text, y_text),
                                (x_text + draw.textsize(char, font)[0],y_text + 15)),fill=add)
                draw.text((x_text, y_text), char, font=font, fill=(0, 0, 0))
            x_text += draw.textsize(char, font)[0]
            char_num += 1
    return np.asarray(image)

def generate_video(generated_texts, name):

    all = []
    gif = []
    
    for i in range(len(generated_texts)-1):
        diff_array = dmp.diff_main(generated_texts[i], generated_texts[i+1])
        dmp.diff_cleanupSemantic(diff_array)
        all.append(diff_array)
    
    for revisions in tqdm(all[:10]):
        gif.append(imageHighlight(revisions))
    
    imageio.mimwrite(name+".gif", gif, "GIF-PIL", duration=1)
    clip = mp.VideoFileClip(name+".gif")
    clip.write_videofile(name+".mp4")

In [ ]:
def main():

    folders = ["llama8_SW_output"]
    seeds = ["seed1", "seed2", "seed3", "seed4"]
    filenames = [f"iter_generation_{i}.txt" for i in range(100)]

    for folder in folders:
        for seed in seeds:
            
            seed_file = open(f"../seeds/{seed}.txt")
            seed_text = seed_file.read()
            generations = [seed_text, seed_text]

            for filename in filenames:
                generation_file = open(f"../outputs/{folder}/{seed}/generation/{filename}")
                generation = generation_file.read()
                generations.append(generation)

            print(folder, seed)
            generate_video(generations, folder+"-"+seed)


if __name__ == '__main__':
    main()